# 05 Introduction to web scraping with BeautifulSoup

## Purpose of this notebook

This notebook introduces the mechanics of web scraping using a simple public web page.

We will use the Victoria University website as a familiar example: <https://www.vu.edu.au/vu-home>

The goal is not to collect a large dataset. The goal is to understand the basic workflow:

1. request one web page
2. check that the request worked
3. turn the HTML text into a BeautifulSoup object
4. find simple tags such as `<title>`, `<h1>`, `<h2>`, and `<a>`
5. pull out readable text
6. save a small result as a CSV file

No regular expressions are used in this notebook. We use the structure of the HTML and normal string methods.


## Before coding: inspect the web page

Before you run the Python code, open the source page in a browser:

<https://www.vu.edu.au/vu-home>

Spend a few minutes checking the page as a person first:

- What is the main heading on the page?
- What section headings can you see?
- What links are visible?
- Is the information public and suitable for a classroom example?

Then check the HTML behind the page. You can use **View page source** or the browser's **Inspect** tool.

Look for simple tags such as:

- `<title>` for the browser page title
- `<h1>` for the main page heading
- `<h2>` for section headings
- `<a>` for links

This step matters because BeautifulSoup does not guess what you want. You need to understand the page structure before asking Python to find tags.


## Scrape responsibly

Before scraping a website, remember:

- use a small number of requests
- identify your request with a clear `User-Agent`
- do not collect private or sensitive information
- prefer an API or official download when one exists
- follow your teacher's instructions for which sites are suitable in class

This notebook makes one request to one public page for learning purposes.


## Step 1: import the libraries

`requests` downloads the page.

`BeautifulSoup` reads the HTML and lets us search for tags.

`pandas` is used at the end to save the result as a CSV file.


In [ ]:
from pathlib import Path

import pandas as pd
import requests
from bs4 import BeautifulSoup


## Step 2: request the page

A website returns a response. The response includes a status code and the page text.

A status code of `200` means the request worked.


In [ ]:
url = "https://www.vu.edu.au/vu-home"
headers = {
    "User-Agent": "VIT1106 teaching example (one polite request)"
}

response = requests.get(url, headers=headers, timeout=30)
print("Status code:", response.status_code)
print("Downloaded characters:", len(response.text))


## Step 3: create a BeautifulSoup object

A web page is HTML text. BeautifulSoup turns that text into a searchable object.

Think of `soup` as a Python view of the page structure. It lets us ask questions such as:

- What is the page title?
- What is the first `<h1>` heading?
- What `<h2>` headings are on the page?
- What links are on the page?


In [ ]:
soup = BeautifulSoup(response.text, "html.parser")
print(type(soup))


## Step 4: read the page title

The browser tab title usually sits inside the `<title>` tag.

`find("title")` returns the first matching tag.

`get_text()` returns the text inside that tag.


In [ ]:
title_tag = soup.find("title")

if title_tag is None:
    print("No title tag found")
else:
    page_title = title_tag.get_text(" ", strip=True)
    print(page_title)


## Step 5: find the main heading

Most pages have one main heading in an `<h1>` tag.


In [ ]:
h1_tag = soup.find("h1")

if h1_tag is None:
    print("No h1 tag found")
else:
    main_heading = h1_tag.get_text(" ", strip=True)
    print(main_heading)


## Step 6: find several section headings

`find_all("h2")` returns a list of all `<h2>` tags.

We can loop through the list and keep the text from each tag.


In [ ]:
h2_tags = soup.find_all("h2")

headings = []
for tag in h2_tags:
    text = tag.get_text(" ", strip=True)
    if text:
        headings.append(text)

print("Number of h2 headings:", len(headings))
headings[:10]


## Step 7: find simple links

Links use the `<a>` tag.

The visible link text is inside the tag. The destination is usually in the `href` attribute.


In [ ]:
link_rows = []

for tag in soup.find_all("a"):
    text = tag.get_text(" ", strip=True)
    href = tag.get("href")

    if text and href:
        link_rows.append({
            "text": text,
            "href": href,
        })

df_links = pd.DataFrame(link_rows)
df_links.head(10)


## Step 8: save a small CSV file

This file is only a small practice output. It shows that scraped text can be saved in a table format.


In [ ]:
DATA_FOLDER = Path("data")
DATA_FOLDER.mkdir(exist_ok=True)

df_headings = pd.DataFrame({"heading": headings})
OUTPUT_FILE = DATA_FOLDER / "vu_home_h2_headings.csv"
df_headings.to_csv(OUTPUT_FILE, index=False)

print("Saved", len(df_headings), "headings to", OUTPUT_FILE)


## What you learned

In this notebook, you practised the basic mechanics of web scraping:

- `requests.get()` downloads a page
- `response.status_code` checks whether the request worked
- `BeautifulSoup(response.text, "html.parser")` parses the HTML
- `soup.find()` gets the first matching tag
- `soup.find_all()` gets all matching tags
- `.get_text(" ", strip=True)` extracts readable text
- `.get("href")` reads an HTML attribute
- `to_csv()` saves the extracted result
